# Changepoint detection — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## A changepoint is evidence that one stable data-generating rule ended and another began
Changepoint detection compares one-regime and two-regime explanations. These examples show mean-shift gain, CUSUM accumulation, penalty control, trend-change fitting, and online delay.

In [ ]:
# 1) split gain: two means fit a level shift better than one mean
y=np.array([10,11,10,18,19,20]); one=np.mean((y-y.mean())**2); left=y[:3]; right=y[3:]; two=(np.sum((left-left.mean())**2)+np.sum((right-right.mean())**2))/len(y); gain=one-two
plt.figure(figsize=(6,3)); plt.plot(y,'o-'); plt.axvline(2.5,color='r',ls='--'); plt.title(f'split gain = {gain:.2f}')
assert abs(gain-18.777777777777782)<1e-12

In [ ]:
# 2) CUSUM accumulates persistent positive surprises
x=np.array([10,11,10,18,19,20]); mu0=10.5; k=0.5; S=0; path=[]
for val in x:
    S=max(0,S+val-mu0-k); path.append(S)
plt.figure(figsize=(6,3)); plt.plot(path,'o-'); plt.axhline(10,color='r',ls='--'); plt.title('CUSUM crosses threshold after sustained shift')
assert np.allclose(path,[0,0,0,7,15,24])

In [ ]:
# 3) penalty prevents every wiggle from becoming a changepoint
gains=np.array([1.2,0.4,3.5,0.9,2.1]); penalties=[0.5,2.0,4.0]; counts=[np.sum(gains>p) for p in penalties]
plt.figure(figsize=(6,3)); plt.bar([str(p) for p in penalties],counts); plt.xlabel('penalty'); plt.ylabel('kept splits'); plt.title('larger penalty keeps fewer changes')
assert counts==[4,2,0]

In [ ]:
# 4) trend changepoint: piecewise line beats one global line
t=np.arange(20); y=np.where(t<10,10+0.2*t,12+0.8*(t-10)); coef=np.polyfit(t,y,1); sse_one=np.sum((y-np.polyval(coef,t))**2); c1=np.polyfit(t[:10],y[:10],1); c2=np.polyfit(t[10:],y[10:],1); sse_two=np.sum((y[:10]-np.polyval(c1,t[:10]))**2)+np.sum((y[10:]-np.polyval(c2,t[10:]))**2)
plt.figure(figsize=(6,3)); plt.plot(t,y,'o'); plt.plot(t,np.polyval(coef,t),label='one line'); plt.axvline(9.5,color='r',ls='--'); plt.legend(); plt.title('piecewise trend removes residual structure')
assert sse_two < 1e-20 and sse_one > 5

In [ ]:
# 5) online detection has delay: require enough evidence before firing
y=np.r_[np.ones(20)*10, np.ones(20)*13] + 0.1*np.random.randn(40); mu=10; S=0; alarm=None
for i,val in enumerate(y):
    S=max(0,S+val-mu-0.5)
    if alarm is None and S>8: alarm=i
plt.figure(figsize=(6,3)); plt.plot(y); plt.axvline(20,color='k',ls='--',label='true change'); plt.axvline(alarm,color='r',ls='--',label='alarm'); plt.legend(); plt.title('alarm comes after evidence accumulates')
assert alarm > 20 and alarm < 26